In [ ]:
import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# PATHS (set here when running via main_pipeline; standalone fallback below)
try:
    openephys_folder
    base_folder
except NameError:
    openephys_folder = Path(r'C:\Users\labuser\Downloads\data for Ilaria\data for Ilaria\2023_09_13\2023-09-13_12-40-06_W3P12_first_piece_Pos2')
    base_folder      = Path(r'C:\Users\labuser\ilaria\Project\processed\openephys_output_2min')
base_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# OPENEPHYS-SPECIFIC PARAMETERS
stream_name = 'Record Node 101#Acquisition_Board-100.Rhythm Data'
# run this to list available streams:
# print(si.get_neo_streams('openephysbinary', openephys_folder))

freq_min = 300.       # highpass cutoff (Hz)

try:
    sorter = SORTER_KS
except NameError:
    sorter = 'tridesclous2'

cref_operator  = 'median'
cref_reference = 'global'

# TEST MODE — clip to this many seconds; None -> use full recording
test_duration_s = None

# TTL-triggered artifact removal
use_ttl_artifacts = False
ttl_channel_id    = 'Rhythm FPGA TTL Input'

# Fallback linear probe pitch (um)
contact_pitch_um = 20.0

# Shared variables exposed to main_pipeline
_source_path     = str(openephys_folder)
_analyzer_sparse = False

In [ ]:
# LOAD
raw_rec = si.read_openephys(openephys_folder, stream_name=stream_name, load_sync_channel=False,
                            block_index=0)

if test_duration_s is not None:
    raw_rec = raw_rec.frame_slice(0, int(test_duration_s * raw_rec.get_sampling_frequency()))
    print(f"[test mode] clipped to {test_duration_s} s")

print(raw_rec)

In [ ]:
# PROBE SETUP
# Attaches a fallback single-column linear probe if no geometry is present.
# contact_pitch_um presets:
#   20 um  —  Neuropixels 1.0 inner column  [default]
#   25 um  —  Neuropixels 2.0
#   50 um  —  Cambridge Neurotech H-series
#  100 um  —  generic low-density silicon probe

if raw_rec.get_property("contact_vector") is None:
    from probeinterface import Probe

    n = raw_rec.get_num_channels()
    probe = Probe(ndim=2, si_units="um")
    probe.set_contacts(
        positions    = np.column_stack([np.zeros(n), np.arange(n, dtype=float) * contact_pitch_um]),
        shapes       = "circle",
        shape_params = {"radius": min(contact_pitch_um * 0.35, 7.5)},
    )
    probe.create_auto_shape(probe_type="tip")
    probe.set_device_channel_indices(np.arange(n))
    raw_rec = raw_rec.set_probe(probe, group_mode="by_probe")
    print(f"[probe] Fallback linear probe attached — {n} ch | {contact_pitch_um} um pitch | {(n-1)*contact_pitch_um:.0f} um span")
else:
    print(f"[probe] Geometry already present — {raw_rec.get_num_channels()} ch, using as-is.")

si.plot_probe_map(raw_rec, with_channel_ids=False)
plt.tight_layout()
plt.show()

In [ ]:
# PREPROCESSING
_n_raw_channels = raw_rec.get_num_channels()

if manual_channel_ids is not None:
    raw_rec = raw_rec.channel_slice(manual_channel_ids)
    print(f"Manual selection: kept {len(manual_channel_ids)} channels")

rec1 = si.highpass_filter(raw_rec, freq_min=freq_min)

_artifact_frames = []
if use_ttl_artifacts:
    _events  = si.read_openephys_event(openephys_folder, block_index=0)
    _ttl_s   = _events.get_event_times(channel_id=ttl_channel_id, segment_index=0)
    _t0      = raw_rec.get_times(segment_index=0)[0]
    _fs      = raw_rec.get_sampling_frequency()
    _artifact_frames.extend(((_ttl_s - _t0) * _fs).astype(np.int64).tolist())
    print(f"TTL artifacts: {len(_ttl_s)} event(s) found")
if artifact_timestamps_s:
    _fs = raw_rec.get_sampling_frequency()
    _artifact_frames.extend([int(t * _fs) for t in artifact_timestamps_s])
if _artifact_frames:
    rec1 = si.remove_artifacts(
        rec1,
        list_triggers=[np.array(sorted(_artifact_frames), dtype=np.int64)],
        ms_before=artifact_ms_before, ms_after=artifact_ms_after,
        mode='zeros',
    )
    print(f"Artifact removal: blanked {len(_artifact_frames)} event(s) total")

bad_channel_ids, _ = si.detect_bad_channels(rec1)

if extra_bad_channel_ids:
    bad_channel_ids = list(set(bad_channel_ids.tolist()) | set(extra_bad_channel_ids))

rec2 = rec1.remove_channels(bad_channel_ids)
rec  = si.common_reference(rec2, operator=cref_operator, reference=cref_reference)
print(rec)